# Evaluating Numerical Expressions

### Table of Contents<a id="toc"></a>

1. [Introduction and Setup](#intro)
2. [Tokenizing and Processing Postfix Expressions](#tokenizing)
3. [Building the Postfix Evaluator](#evaluator)
4. [Handling Precedence and Parentheses](#precedence)
5. [The Shunting-Yard Algorithm: Infix to Postfix](#algorithm)
6. [Conclusion](#conclusion)

## Introduction and Setup
<a id="intro"></a>
When humans look at a mathematical expression like $3 + 4 \times 2$, we intuitively understand the order of operations (multiplication happens before addition). However, computers read expressions linearly and require structured rules to parse them correctly.

This project implements a classic computer science workflow for evaluating mathematical strings. We start by using a **Stack** data structure to evaluate expressions written in **Postfix notation** (where operators follow their operands, like `3 4 2 * +`). Later, we build a translator that converts standard human-readable **Infix notation** (like `3 + (4 * 2)`) into Postfix notation using the **Shunting-Yard algorithm**, allowing us to accurately compute complex numerical strings while respecting standard mathematical rules and parentheses.

To process our mathematical expressions, we need a specific type of data structure called a **Stack**. Think of a stack like a pile of dinner plates: you can only add a new plate to the very top (**push**), look at the top plate (**peek**), or remove the top plate (**pop**). This "Last-In, First-Out" behavior is perfect for tracking numbers and operations as we read through an equation. Here, we build our custom `Stack` by building on top of a previously defined `LinkedList`.

In [1]:
from linked_list import LinkedList

class Stack(LinkedList):
    
    def push(self, data):
        self.append(data)

    def peek(self):
        return self.tail.data

    def pop(self):
        ret = self.tail.data
        if self.length == 1:
            self.tail = self.head = None
        else:
            self.tail = self.tail.prev
            self.tail.next = None
        self.length -= 1
        return ret

[Back to Table of Contents](#toc)

## Tokenizing and Processing Postfix Expressions
<a id="tokenizing"></a>
Before calculating anything, the computer needs to break a long string of characters into individual, manageable pieces—a process called **tokenization**. This simple function splits a single string of text into a list of individual numbers and mathematical symbols wherever it finds a space.

In [2]:
def tokenize(string):
    return string.split()

print(tokenize("12 2 4 + / 21 *"))

['12', '2', '4', '+', '/', '21', '*']


Running this function on `"12 2 4 + / 21 *"` breaks it down into a clean, ordered Python list of individual string elements: `['12', '2', '4', '+', '/', '21', '*']`.

In Postfix notation, when we encounter a mathematical operator (like `+` or `-`), it applies to the last two numbers we looked at. These functions define how each specific operation behaves. For any given operator, the program pops the top two numbers off our stack, performs the math (making sure to keep the order correct for subtraction, division, and exponents), and pushes the final answer right back onto the stack to be used in the next step.

In [3]:
def process_minus(stack):
    top = stack.pop()
    second_to_top = stack.pop()
    result = second_to_top - top
    stack.push(result)

def process_plus(stack):
    top = stack.pop()
    second_to_top = stack.pop()
    result = second_to_top + top
    stack.push(result)

def process_times(stack):
    top = stack.pop()
    second_to_top = stack.pop()
    result = second_to_top * top
    stack.push(result)

def process_divide(stack):
    top = stack.pop()
    second_to_top = stack.pop()
    result = second_to_top / top
    stack.push(result)

def process_pow(stack):
    top = stack.pop()
    second_to_top = stack.pop()
    result = second_to_top ** top
    stack.push(result)

[Back to Table of Contents](#toc)

## Building the Postfix Evaluator
<a id="evaluator"></a>
This is the central engine for solving Postfix equations. It loops through our list of separated tokens one by one. If it encounters a number, it pushes it onto the stack. If it encounters an operator, it triggers the matching math function we defined earlier to combine the top two numbers. By the time it reaches the end of the list, all the math is complete, and the final answer is left sitting alone on the stack, ready to be returned.

In [4]:
def evaluate_postfix(expression):
    stack = Stack()
    tokens = tokenize(expression)
    for token in tokens:
        if token == '+':
            process_plus(stack)
        elif token == '-':
            process_minus(stack)
        elif token == '*':
            process_times(stack)
        elif token == '/':
            process_divide(stack)
        elif token == '**':
            process_pow(stack)
        else:
            stack.push(float(token))
    return stack.pop()

Now it's time to put our Postfix engine to the test with a variety of equations, ranging from simple subtractions to highly nested, multi-step calculations.

In [5]:
expressions = [
    "4 6 -",
    "4 1 2 9 3 / * + 5 - *",
    "1 2 + 3 -",
    "1 2 - 3 +",
    "10 3 5 * 16 4 - / +",
    "5 3 4 2 - ** *",
    "12 2 4 + / 21 *",
    "1 1 + 2 **",
    "1 1 2 ** +"
]

for expression in expressions:
    answer = evaluate_postfix(expression)
    print(answer)

-2.0
8.0
0.0
2.0
11.25
45.0
42.0
4.0
2.0


**Output Results:** The program successfully processes all nine test cases, printing out their final calculated decimal values:

* `"4 6 -"` correctly results in `-2.0`.
* Complex lines like `"4 1 2 9 3 / * + 5 - *"` resolve neatly to `8.0`.
* Exponent tests like `"1 1 + 2 **"` (which means $(1+1)^2$) accurately yield `4.0`.

[Back to Table of Contents](#toc)

## Handling Precedence and Parentheses
<a id="precedence"></a>
Standard human math doesn't just move from left to right; it follows strict hierarchy rules (PEMDAS/BODMAS). To teach our program these rules, we create a dictionary that assigns a numerical "importance tier" to each operator. Exponents (`**`) get the highest priority, multiplication and division come next, and addition and subtraction sit at the bottom.

In [6]:
precedence = {
    "+": 1, 
    "-": 1,
    "*": 2,
    "/": 2,
    "**": 3
}


print(precedence["+"] < precedence["*"])
print(precedence["+"] < precedence["-"])
print(precedence["/"] < precedence["**"])

True
False
True


**Output Result:** The boolean checks confirm our logic is sound: addition has lower precedence than multiplication (`True`), addition and subtraction are equal (`False`, because they have the same priority level), and division yields to exponents (`True`).

Parentheses are used to manually override standard math priorities. These two functions dictate how our program isolates grouped calculations. When an opening parenthesis `(` is found, it is saved on the stack as a temporary divider. When a closing parenthesis `)` is hit, the program immediately flushes out and completes all the mathematical operations that were trapped inside those brackets, wrapping up that isolated sub-equation before moving on.

In [7]:
def process_opening_parenthesis(stack):
    stack.push("(")

In [8]:
def process_closing_parenthesis(stack, postfix):
    while stack.peek() != "(":
        postfix.append(stack.pop())
    # Remove the opening bracket
    stack.pop()

These helper functions sort our incoming data. Numbers are simple—they skip the line and go straight to the final output list. Operators, however, must wait on the stack. Before an operator sits down on the stack, it checks to see if the operator already at the top has a higher or equal mathematical priority. If it does, that existing operator gets pushed out to the final list first, ensuring correct order of operations.

In [9]:
def process_operator(stack, postfix, operator):
    while len(stack) > 0 and stack.peek() in precedence and precedence[stack.peek()] >= precedence[operator]:
        postfix.append(stack.pop())
    stack.push(operator)

In [10]:
def process_number(postfix, number):
    postfix.append(number)

[Back to Table of Contents](#toc)

## The Shunting-Yard Algorithm: Infix to Postfix
<a id="algorithm"></a>
Here, we assemble the **Shunting-Yard algorithm** (`infix_to_postfix`). This function takes a standard math expression, sorts it through our parenthesized and operator-priority rules, and outputs a cleanly formatted Postfix string. The `evaluate` function ties everything together: it takes a normal human equation, translates it behind the scenes into Postfix, and passes it to our evaluation engine to get the final answer.

In [11]:
def infix_to_postfix(expression):
    tokens = tokenize(expression)
    stack = Stack()
    postfix = []
    for token in tokens:
        if token == "(":
            process_opening_parenthesis(stack)
        elif token == ")":
            process_closing_parenthesis(stack, postfix)
        elif token in precedence:
            process_operator(stack, postfix, token)
        else:
            process_number(postfix, token)
    while len(stack) > 0:
        postfix.append(stack.pop())
    return " ".join(postfix)

In [12]:
def evaluate(expression):
    postfix_expression = infix_to_postfix(expression)
    return evaluate_postfix(postfix_expression)

This final test suite challenges our completed pipeline with standard infix algebraic notations, utilizing varying levels of nested parentheses, division chaining, and exponentiation priorities.

In [13]:
expressions = [
    "1 + 1",
    "1 * ( 2 - ( 1 + 1 ) )",
    "4 * ( 1 + 2 * ( 9 / 3 ) - 5 )",
    "10 + 3 * 5 / ( 16 - 4 * 1 )",
    "2 * 2 * 2 * 2 * 2 * 2 * 2 * 2",
    "2 ** 2 ** 2 ** 2 ** 2",
    "( 1 - 2 ) / ( 3 - 5 )",
    "9 / 8 * 8",
    "64 / ( 8 * 8 )",
]

for expression in expressions:
    answer = evaluate(expression)
    print(answer)

2.0
0.0
8.0
11.25
256.0
65536.0
0.5
9.0
1.0


**Output Results:**
The system perfectly evaluates the human-readable expressions:

* `"1 + 1"` yields `2.0`.
* Overriding priorities with parentheses, such as `"1 * ( 2 - ( 1 + 1 ) )"`, correctly results in `0.0`.
* The exponent chain `"2  2  2  2  2"` evaluates to `65536.0`, matching the left-to-right processing of our standard loop implementation.

## Conclusion
<a id="conclusion"></a>
Throughout this project, we successfully designed and implemented an end-to-end numerical expression parser. By utilizing a linear **Stack** data structure, we bypassed the computational ambiguities of human mathematical notation. We demonstrated how complex order-of-operations logic, including bracket overrides, can be perfectly mapped using the **Shunting-Yard algorithm** to translate equations into a computer-friendly Postfix format. The result is a robust, lightweight evaluation engine capable of reliably calculating intricate arithmetic expressions.

[Back to Table of Contents](#toc)